In [3]:
import pandas as pd

df_contrat = pd.read_excel('Données_Assurance_S1.2_S2.2.xlsx', sheet_name='Contrats')
df_contrat.head()

,REF_PERSONNE,NUM_CONTRAT,LIB_PRODUIT,EFFET_CONTRAT,DATE_EXPIRATION,PROCHAIN_TERME,LIB_ETAT_CONTRAT,branche,somme_quittances,statut_paiement,Capital_assure
0,122335,1996101000186,TEMPORAIRE DECES A CAPITAL DECROISSANT LINEAIR...,1996-06-06 00:00:00.000,2017-06-05 00:00:00.000,NaN,EXPIRE,VIE,630.0,Non payé,10000.000
1,115271,1996103000612,ASSURANCE VIE COMPLEMENT RETRAITE - HORIZON (AS),1996-09-16 00:00:00.000,2024-09-15 00:00:00.000,NaN,EXPIRE,VIE,8568.0,Payé,4038.000
2,115197,1996102000382,MIXTE REVALORISABLE (AS),1996-09-03 00:00:00.000,2025-09-03 00:00:00.000,NaN,RESILIE,VIE,30.9,Payé,4810.969
3,122355,1996101000747,TEMPORAIRE DECES A CAPITAL DECROISSANT LINEAIR...,1996-10-11 00:00:00.000,2017-10-10 00:00:00.000,NaN,EXPIRE,VIE,462.0,Non payé,10000.000
4,115302,1996103001078,ASSURANCE VIE COMPLEMENT RETRAITE - HORIZON (AS),1996-10-15 00:00:00.000,2030-10-14 00:00:00.000,NaN,EN COURS,VIE,10753.2,Non payé,6012.000


In [5]:
import pandas as pd
from datetime import datetime, timedelta


# Conversion des dates
df_contrat["DATE_EXPIRATION"] = pd.to_datetime(df_contrat["DATE_EXPIRATION"], errors="coerce")
today = datetime.now()
six_months = today + timedelta(days=180)

# ---- Opportunité ----
def detect_opportunite(row):
    if row["LIB_ETAT_CONTRAT"] == "RESILIE":
        return "Proposer produit alternatif"
    elif row["DATE_EXPIRATION"] < today and row["LIB_ETAT_CONTRAT"] != "RESILIE":
        return "Contrat échu – proposer renouvellement"
    elif today <= row["DATE_EXPIRATION"] <= six_months:
        return "À échéance bientôt – proposer upsell"
    else:
        return "Contrat actif – suivi normal"

df_contrat["opportunite"] = df_contrat.apply(detect_opportunite, axis=1)

# ---- Action paiement ----
def detect_action_paiement(row):
    if row["statut_paiement"] == "Non payé" and row["LIB_ETAT_CONTRAT"] == "EN COURS":
        return "Relance paiement / proposer facilités"
    elif row["statut_paiement"] == "Non payé" and row["LIB_ETAT_CONTRAT"] in ["EXPIRE","RESILIE"]:
        return "Proposer nouvelle offre adaptée"
    else:
        return "Paiement OK"

df_contrat["action_paiement"] = df_contrat.apply(detect_action_paiement, axis=1)

# Résultat
print(df_contrat[["NUM_CONTRAT","LIB_PRODUIT","LIB_ETAT_CONTRAT","DATE_EXPIRATION","statut_paiement","opportunite","action_paiement"]])


         NUM_CONTRAT                                        LIB_PRODUIT  \
0      1996101000186  TEMPORAIRE DECES A CAPITAL DECROISSANT LINEAIR...   
1      1996103000612   ASSURANCE VIE COMPLEMENT RETRAITE - HORIZON (AS)   
2      1996102000382                           MIXTE REVALORISABLE (AS)   
3      1996101000747  TEMPORAIRE DECES A CAPITAL DECROISSANT LINEAIR...   
4      1996103001078   ASSURANCE VIE COMPLEMENT RETRAITE - HORIZON (AS)   
...              ...                                                ...   
92976  2025101051482  TEMPORAIRE DECES A CAPITAL DECROISSANT LINEAIR...   
92977  2025301076198                                         AUTOMOBILE   
92978  2022301023338                         OMNICANAL AUTOMOBILE PACK2   
92979  2025206024922              ASSISTANCES EN VOYAGES - PLAN BASIQUE   
92980  2019301003288                                         AUTOMOBILE   

      LIB_ETAT_CONTRAT DATE_EXPIRATION statut_paiement  \
0               EXPIRE      2017-06-05   

In [ ]:
from gpt4all import GPT4All
import smtplib
from email.mime.text import MIMEText
from datetime import datetime

# Charger le modèle local
model = GPT4All("gpt4all-falcon-newbpe-q4_0.gguf")

# Fonction génération email avec GPT4All
def generate_email_with_gpt4all(row):
    if row["opportunite"] == 'Proposer produit alternatif':


    else:
        prompt = f"""
    Tu es un assistant en assurance. Génère un email professionnel et personnalisé
    pour le client concernant son contrat.

    Détails du contrat :
    - Numéro : {row['NUM_CONTRAT']}
    - Produit : {row['LIB_PRODUIT']}
    - État : {row['LIB_ETAT_CONTRAT']}
    - Date d'expiration : {row['DATE_EXPIRATION'].date()}
    - Statut paiement : {row['statut_paiement']}

    Le mail doit être courtois, clair, et contenir un appel à l’action (renouveler, payer, découvrir une offre alternative...).
    Commence par 'Bonjour,'.
    Finis par 'Cordialement, Votre compagnie d'assurance'.
        """

        email_text = model.generate(prompt, max_tokens=300)
        return email_text

mail = generate_email_with_gpt4all(df_contrat.iloc[0])
print(mail)



Subject: Important Update on Your Temporary Death Coverage Plan

Dear [Client Name],

I hope this email finds you well. I am writing to inform you that your temporary death coverage plan with our company is set to expire on June 5th, 2017. As a reminder, this plan was designed to provide financial support in the event of a sudden and unexpected death during the term of your policy.

As of today, your policy has expired and we have not received any payment for your coverage. We would like to remind you that it is important to keep your policy up-to-date and make timely payments to ensure that you are protected in the event of a loss.

We understand that unexpected events can occur, and we want to assure you that our company is committed to providing excellent customer service and support throughout the claims process. If you have any questions or concerns regarding your coverage, please do not hesitate to contact us at [Phone Number] or visit our website at [Website URL].

We hope this

In [7]:
print(df_contrat.iloc[0])


REF_PERSONNE                                                   122335
NUM_CONTRAT                                             1996101000186
LIB_PRODUIT         TEMPORAIRE DECES A CAPITAL DECROISSANT LINEAIR...
EFFET_CONTRAT                                 1996-06-06 00:00:00.000
DATE_EXPIRATION                                   2017-06-05 00:00:00
PROCHAIN_TERME                                                    NaN
LIB_ETAT_CONTRAT                                               EXPIRE
branche                                                           VIE
somme_quittances                                                630.0
statut_paiement                                              Non payé
Capital_assure                                                10000.0
opportunite                    Contrat échu – proposer renouvellement
action_paiement                       Proposer nouvelle offre adaptée
Name: 0, dtype: object
